# 00 — ZHVI Smoothing Check

**Purpose.** Answers the open question flagged in `docs/Housing_Price_Decline_Team_Reference.md`'s
ZHVI Smoothing Verification section: is Zillow's "smoothed" ZHVI cut a *trailing*
filter (uses only past months) or a *centered* one (also uses future months)?
If it were centered, every momentum feature in this project (1, 3, and 6-month
lagged ZHVI percent change) would carry a lookahead leak baked into it, all
the way back to 2000, since the smoothing method doesn't change over the
series.

This notebook runs the three checks -- national
turning points, a data-selected county's turning points, and a full-history
cross-correlation sweep -- against **this project's own** downloaded ZHVI
file, so the leakage-safety claim for this project's momentum features
rests on evidence gathered here, not borrowed from elsewhere.

**Method: empirical, not documentation-based.** Zillow's own published
methodology doesn't state a smoothing window length or direction -- "smoothed"
is described qualitatively, not as a specified N-month filter. So this
notebook tests the downloaded data directly instead of reading documentation.

**One confound stated up front.** Only two ZHVI variants exist for this tier
and segment: raw/non-seasonally-adjusted (NSA) and smoothed+seasonally-adjusted
(SA). There is no raw+SA variant available, so this comparison can't fully
isolate the smoothing effect from the seasonal-adjustment effect -- both
differences are bundled into the same comparison. The turning-point checks
below (does the smoothed series ever *lead* the raw one) are the stronger
evidence for that reason: seasonal adjustment on its own has no reason to
make a series lead its own unadjusted version, so a leading smoothed series
would be the specific signature of a lookahead component in the smoothing
step. The cross-correlation sweep is the weaker, secondary check -- it shows
whether the filter *leans* trailing, not that it contains literally zero
forward-looking weight.

**No conclusion is asserted in advance.** Section 8 below computes its
verdict programmatically from whatever this run's numbers actually are --
run this notebook to get the real answer for this project's data, don't
take the direction of the result for granted from the sibling project.


## 1. Setup

In [1]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

import os
import json
import requests
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)


Python executable: /Users/<user>/code/jolennd/bootcamp/neueFische/projects/Housing-Risk-Teamwork/.venv/bin/python
Python version: 3.14.7 (v3.14.7:823f0323ee6, Aug  5 2026, 07:07:01) [Clang 21.0.0 (clang-2100.1.1.101)]


## 2. Download the raw (non-seasonally-adjusted, non-smoothed) ZHVI cut

The smoothed+SA file (`County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv`)
is already downloaded at `../data/raw/zillow/zhvi_county.csv` from notebook 01.
This cell downloads its raw/NSA counterpart -- same tier (Middle Tier,
33rd-67th percentile), same segment (SFR + Condo/Co-op), same geography
(County) -- the only difference in the filename is the missing `_sm_sa`.

This is also where the "does the locally cached smoothed file still match
the live host's current version" provenance check happens, since we're
already talking to the same endpoint.


In [2]:
ZILLOW_RAW_URL = (
    "https://files.zillowstatic.com/research/public_csvs/zhvi/"
    "County_zhvi_uc_sfrcondo_tier_0.33_0.67_month.csv"
)
ZILLOW_SMOOTHED_URL = (
    "https://files.zillowstatic.com/research/public_csvs/zhvi/"
    "County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
)

RAW_DIR = "../data/raw/zillow"
os.makedirs(RAW_DIR, exist_ok=True)

raw_nsa_path = os.path.join(RAW_DIR, "zhvi_county_raw_nsa.csv")
smoothed_path = os.path.join(RAW_DIR, "zhvi_county.csv")  # from notebook 01

if not os.path.exists(raw_nsa_path):
    resp = requests.get(ZILLOW_RAW_URL, timeout=60)
    resp.raise_for_status()
    with open(raw_nsa_path, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content):,} bytes to {raw_nsa_path}")
else:
    print(f"Using cached file at {raw_nsa_path}")

assert os.path.exists(smoothed_path), (
    f"{smoothed_path} not found -- run 01_zillow_ingest_clean.ipynb first "
    "(it downloads the smoothed+SA file this notebook compares against)."
)

# Provenance check: does the live host's current smoothed file still match
# what's cached locally? (file size is a cheap, effective proxy here)
head_resp = requests.head(ZILLOW_SMOOTHED_URL, timeout=30)
host_size = int(head_resp.headers.get("Content-Length", -1))
local_size = os.path.getsize(smoothed_path)
if host_size == local_size:
    print(f"Live host smoothed-file size ({host_size:,} bytes) matches the local cached copy -- still current.")
else:
    print(
        f"NOTE: live host smoothed-file size ({host_size:,} bytes) differs from the "
        f"local cached copy ({local_size:,} bytes) -- Zillow has published a newer "
        "file since 01_zillow_ingest_clean.ipynb was last run. Not a problem for "
        "this notebook, but worth re-running notebook 01 to pick up the update."
    )


Using cached file at ../data/raw/zillow/zhvi_county_raw_nsa.csv
Live host smoothed-file size (13,579,570 bytes) matches the local cached copy -- still current.


## 3. Reshape both variants to long format

Same melt-and-build-FIPS approach as notebook 01: wide date columns ->
long `county_fips` / `month` / `zhvi` rows, trimmed to each county's own
first-to-last published month so a county's pre-coverage or not-yet-published
NaNs don't get compared as if they were real gaps.


In [3]:
def load_and_reshape(path, value_col):
    wide = pd.read_csv(path)
    wide["county_fips"] = (
        wide["StateCodeFIPS"].astype(str).str.zfill(2)
        + wide["MunicipalCodeFIPS"].astype(str).str.zfill(3)
    )
    date_cols = [c for c in wide.columns if c[:4].isdigit()]
    long_df = wide.melt(
        id_vars=["county_fips", "SizeRank", "RegionName"],
        value_vars=date_cols,
        var_name="month",
        value_name=value_col,
    )
    long_df["month"] = pd.to_datetime(long_df["month"])

    # Trim each county to its own first-to-last non-null month (leading/
    # trailing NaNs from coverage start/publish lag, not real internal gaps).
    # Vectorized via a merge onto per-county (min, max) valid-month bounds --
    # NOT groupby(...).apply(trim): recent pandas silently drops the grouping
    # column ("county_fips") from the reassembled frame in that pattern when
    # the applied function returns the group largely unchanged for most
    # groups, which is exactly the KeyError this hit. A county with zero
    # valid months simply has no bounds row, so the inner merge below drops
    # it entirely, same effect as the old trim()'s empty-group branch.
    valid = long_df[long_df[value_col].notna()]
    county_bounds = valid.groupby("county_fips")["month"].agg(["min", "max"])
    long_df = long_df.merge(county_bounds, on="county_fips", how="inner")
    long_df = long_df[
        (long_df["month"] >= long_df["min"]) & (long_df["month"] <= long_df["max"])
    ].drop(columns=["min", "max"])

    return long_df.sort_values(["county_fips", "month"]).reset_index(drop=True)

raw_long = load_and_reshape(raw_nsa_path, "zhvi_raw")
smoothed_long = load_and_reshape(smoothed_path, "zhvi_smoothed")

print(f"Raw/NSA:        {len(raw_long):,} county-month rows, {raw_long['county_fips'].nunique():,} counties")
print(f"Smoothed+SA:    {len(smoothed_long):,} county-month rows, {smoothed_long['county_fips'].nunique():,} counties")

both = raw_long.merge(
    smoothed_long[["county_fips", "month", "zhvi_smoothed"]],
    on=["county_fips", "month"],
    how="inner",
)
print(f"Matched on both: {len(both):,} county-month rows")
both.head()


/var/folders/v8/pbmz67g97jd1djd3tpb20jzh0000gn/T/ipykernel_23061/3435381485.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wide["county_fips"] = (
/var/folders/v8/pbmz67g97jd1djd3tpb20jzh0000gn/T/ipykernel_23061/3435381485.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wide["county_fips"] = (


Raw/NSA:        854,333 county-month rows, 3,071 counties
Smoothed+SA:    709,996 county-month rows, 3,071 counties
Matched on both: 709,996 county-month rows


,county_fips,SizeRank,RegionName,month,zhvi_raw,zhvi_smoothed
0,01001,904,Autauga County,2000-01-31,121979.588489,123634.685391
1,01001,904,Autauga County,2000-02-29,122145.289320,123665.582501
2,01001,904,Autauga County,2000-03-31,121797.257356,123462.740267
3,01001,904,Autauga County,2000-04-30,122649.497574,123385.784631
4,01001,904,Autauga County,2000-05-31,123610.028067,123383.406204


## 4. National aggregate turning points (2021-2024)

Neither file ships a national aggregate row (only per-county rows are
present), so one is constructed here: each month's cross-county **median**
percent change is chained into a synthetic national index, separately for
the raw and the smoothed series. Turning points (local peaks and troughs)
are then located in each constructed index within the 2021-2024 window and
compared month-for-month.

If the smoothing filter is trailing, the smoothed series' turning points
should land on or after the raw series' -- never before it.


In [4]:
def national_index(long_df, value_col):
    pct_change = (
        long_df.sort_values(["county_fips", "month"])
        .groupby("county_fips")[value_col]
        .pct_change()
    )
    monthly_median_change = pct_change.groupby(long_df["month"]).median()
    monthly_median_change = monthly_median_change.sort_index().fillna(0)
    national = (1 + monthly_median_change).cumprod()
    national.name = value_col
    return national

national_raw = national_index(raw_long, "zhvi_raw")
national_smoothed = national_index(smoothed_long, "zhvi_smoothed")

def find_turning_points(series, start, end):
    """Local maxima and minima within [start, end], by simple neighbor comparison."""
    window = series.loc[start:end]
    peaks, troughs = [], []
    values = window.values
    idx = window.index
    for i in range(1, len(values) - 1):
        if values[i] > values[i - 1] and values[i] > values[i + 1]:
            peaks.append(idx[i])
        elif values[i] < values[i - 1] and values[i] < values[i + 1]:
            troughs.append(idx[i])
    return peaks, troughs

WINDOW_START, WINDOW_END = "2021-01-01", "2024-12-31"
raw_peaks, raw_troughs = find_turning_points(national_raw, WINDOW_START, WINDOW_END)
smoothed_peaks, smoothed_troughs = find_turning_points(national_smoothed, WINDOW_START, WINDOW_END)

print("Raw national index -- peaks:", [d.date() for d in raw_peaks])
print("Raw national index -- troughs:", [d.date() for d in raw_troughs])
print("Smoothed national index -- peaks:", [d.date() for d in smoothed_peaks])
print("Smoothed national index -- troughs:", [d.date() for d in smoothed_troughs])

national_turning_point_results = {
    "raw_peaks": [str(d.date()) for d in raw_peaks],
    "raw_troughs": [str(d.date()) for d in raw_troughs],
    "smoothed_peaks": [str(d.date()) for d in smoothed_peaks],
    "smoothed_troughs": [str(d.date()) for d in smoothed_troughs],
}


Raw national index -- peaks: [datetime.date(2022, 7, 31), datetime.date(2023, 8, 31), datetime.date(2024, 8, 31)]
Raw national index -- troughs: [datetime.date(2023, 2, 28), datetime.date(2024, 1, 31)]
Smoothed national index -- peaks: [datetime.date(2022, 8, 31), datetime.date(2023, 11, 30)]
Smoothed national index -- troughs: [datetime.date(2023, 2, 28), datetime.date(2024, 1, 31)]


## 5. A specific county, selected by the data

Rather than picking a county from memory, this selects the county with the
single sharpest one-month percent decline in the **raw** series during
2022-2023, among the 100 largest US counties by `SizeRank`. Its raw vs.
smoothed peak and trough dates are then compared the same way as the
national series above.


In [5]:
top100_fips = (
    raw_long[["county_fips", "SizeRank"]]
    .drop_duplicates()
    .nsmallest(100, "SizeRank")["county_fips"]
)

window_mask = (raw_long["month"] >= "2022-01-01") & (raw_long["month"] <= "2023-12-31")
top100_window = raw_long[raw_long["county_fips"].isin(top100_fips) & window_mask].copy()
top100_window = top100_window.sort_values(["county_fips", "month"])
top100_window["pct_change"] = top100_window.groupby("county_fips")["zhvi_raw"].pct_change()

sharpest_row = top100_window.loc[top100_window["pct_change"].idxmin()]
selected_fips = sharpest_row["county_fips"]
selected_name = raw_long.loc[raw_long["county_fips"] == selected_fips, "RegionName"].iloc[0]

print(
    f"Selected county: {selected_name} (FIPS {selected_fips}), "
    f"sharpest single-month raw decline: {sharpest_row['pct_change']:.2%} "
    f"in {sharpest_row['month'].date()}"
)

county_raw = raw_long.loc[raw_long["county_fips"] == selected_fips].set_index("month")["zhvi_raw"]
county_smoothed = smoothed_long.loc[smoothed_long["county_fips"] == selected_fips].set_index("month")["zhvi_smoothed"]

county_raw_peaks, county_raw_troughs = find_turning_points(county_raw, WINDOW_START, WINDOW_END)
county_smoothed_peaks, county_smoothed_troughs = find_turning_points(county_smoothed, WINDOW_START, WINDOW_END)

print("County raw -- peaks:", [d.date() for d in county_raw_peaks], "troughs:", [d.date() for d in county_raw_troughs])
print("County smoothed -- peaks:", [d.date() for d in county_smoothed_peaks], "troughs:", [d.date() for d in county_smoothed_troughs])

county_turning_point_results = {
    "selected_county_fips": selected_fips,
    "selected_county_name": selected_name,
    "sharpest_decline_pct": float(sharpest_row["pct_change"]),
    "sharpest_decline_month": str(sharpest_row["month"].date()),
    "raw_peaks": [str(d.date()) for d in county_raw_peaks],
    "raw_troughs": [str(d.date()) for d in county_raw_troughs],
    "smoothed_peaks": [str(d.date()) for d in county_smoothed_peaks],
    "smoothed_troughs": [str(d.date()) for d in county_smoothed_troughs],
}


Selected county: Santa Clara County (FIPS 06085), sharpest single-month raw decline: -4.32% in 2022-08-31
County raw -- peaks: [datetime.date(2021, 7, 31), datetime.date(2022, 5, 31), datetime.date(2023, 10, 31), datetime.date(2024, 6, 30)] troughs: [datetime.date(2021, 9, 30), datetime.date(2023, 1, 31), datetime.date(2024, 1, 31), datetime.date(2024, 10, 31)]
County smoothed -- peaks: [datetime.date(2022, 5, 31), datetime.date(2024, 1, 31)] troughs: [datetime.date(2023, 5, 31), datetime.date(2024, 2, 29)]


## 6. Full-history cross-correlation sweep (month-over-month growth)

Pools every county-month pair across the full matched history and
correlates the smoothed series' growth at month *t* against the raw
series' growth at *t-k* (past) and *t+k* (future), for k = 1 through 6
months. A trailing filter should correlate more strongly with **past**
raw growth than with the equivalent **future** raw growth, at every
tested lag. A centered, symmetric filter would show equal correlation in
both directions.


In [6]:
panel = both.sort_values(["county_fips", "month"]).copy()
panel["raw_growth"] = panel.groupby("county_fips")["zhvi_raw"].pct_change()
panel["smoothed_growth"] = panel.groupby("county_fips")["zhvi_smoothed"].pct_change()
panel = panel.dropna(subset=["raw_growth", "smoothed_growth"])

lag_correlations = {}
for k in range(0, 7):
    # past: smoothed(t) vs raw(t-k)  -- shift raw_growth forward by k so row t holds raw(t-k)
    past_raw = panel.groupby("county_fips")["raw_growth"].shift(k)
    past_corr = panel["smoothed_growth"].corr(past_raw)

    # future: smoothed(t) vs raw(t+k) -- shift raw_growth backward by k so row t holds raw(t+k)
    future_raw = panel.groupby("county_fips")["raw_growth"].shift(-k)
    future_corr = panel["smoothed_growth"].corr(future_raw)

    lag_correlations[k] = {"past_corr": float(past_corr), "future_corr": float(future_corr)}
    if k == 0:
        print(f"lag 0 (same month):  corr = {past_corr:.4f}")
    else:
        print(f"lag {k} past (raw t-{k}): corr = {past_corr:.4f}   |   lag {k} future (raw t+{k}): corr = {future_corr:.4f}")

asymmetric_toward_past = all(
    lag_correlations[k]["past_corr"] > lag_correlations[k]["future_corr"]
    for k in range(1, 7)
)
print(f"\nAsymmetric toward the past at every tested lag (1-6 months): {asymmetric_toward_past}")


lag 0 (same month):  corr = 0.6450
lag 1 past (raw t-1): corr = 0.6816   |   lag 1 future (raw t+1): corr = 0.4344
lag 2 past (raw t-2): corr = 0.6027   |   lag 2 future (raw t+2): corr = 0.3262
lag 3 past (raw t-3): corr = 0.3722   |   lag 3 future (raw t+3): corr = 0.2563
lag 4 past (raw t-4): corr = 0.2650   |   lag 4 future (raw t+4): corr = 0.2029
lag 5 past (raw t-5): corr = 0.2155   |   lag 5 future (raw t+5): corr = 0.1612
lag 6 past (raw t-6): corr = 0.1978   |   lag 6 future (raw t+6): corr = 0.1355

Asymmetric toward the past at every tested lag (1-6 months): True


## 7. The bundled confound, restated

As noted in Section 0: only raw/NSA and smoothed/SA variants exist for this
tier, there's no raw+SA file to isolate smoothing from seasonal adjustment
cleanly. Seasonal adjustment on its own has no mechanism to make a series
*lead* its own unadjusted version, so Section 4 and 5's turning-point checks
(does smoothed ever lead raw) are the stronger evidence here. Section 6's
correlation asymmetry shows the filter *leans* trailing; it doesn't by
itself prove zero forward-looking weight, only that any such weight is
small relative to the trailing component.


## 8. Verdict, computed from this run's numbers

No conclusion is assumed going in -- this cell checks the actual results
above and prints a verdict based on them, so whatever this run finds is
what's reported, not what the sibling project found.


In [7]:
def series_ever_leads(peaks, troughs, other_peaks, other_troughs):
    """True if this series has any peak/trough strictly earlier than every
    corresponding point in the other series (a leading, i.e. lookahead, signature)."""
    import pandas as pd
    for p in peaks:
        if other_peaks and pd.Timestamp(p) < min(pd.Timestamp(x) for x in other_peaks):
            return True
    for t in troughs:
        if other_troughs and pd.Timestamp(t) < min(pd.Timestamp(x) for x in other_troughs):
            return True
    return False

smoothed_leads_nationally = series_ever_leads(
    smoothed_peaks, smoothed_troughs, raw_peaks, raw_troughs
)
smoothed_leads_county = series_ever_leads(
    county_smoothed_peaks, county_smoothed_troughs, county_raw_peaks, county_raw_troughs
)

no_leading_evidence = not smoothed_leads_nationally and not smoothed_leads_county

if no_leading_evidence and asymmetric_toward_past:
    verdict = (
        "TRAILING, not centered. The smoothed series never leads the raw series "
        "at either the national or the selected-county level (it only matches or "
        "lags), and the full-history correlation sweep is asymmetric toward the "
        "past at every tested lag. This is the empirical signature of a trailing "
        "filter. No evidence of a dominant lookahead component was found -- see "
        "Section 7's confound note for the residual uncertainty a bundled "
        "smoothing/seasonal-adjustment comparison can't fully rule out."
    )
elif no_leading_evidence and not asymmetric_toward_past:
    verdict = (
        "LIKELY TRAILING, but weaker evidence than expected. No turning point "
        "leads the raw series, but the correlation sweep is NOT consistently "
        "asymmetric toward the past -- investigate further before relying on "
        "this as confirmation."
    )
else:
    verdict = (
        "STOP -- possible centered/lookahead component found. The smoothed "
        "series led the raw series at at least one turning point "
        f"(nationally: {smoothed_leads_nationally}, selected county: "
        f"{smoothed_leads_county}). This would mean the momentum features in "
        "this project may carry a leakage risk from ZHVI smoothing itself -- "
        "do not treat momentum as leakage-safe until this is investigated "
        "further."
    )

print(verdict)


TRAILING, not centered. The smoothed series never leads the raw series at either the national or the selected-county level (it only matches or lags), and the full-history correlation sweep is asymmetric toward the past at every tested lag. This is the empirical signature of a trailing filter. No evidence of a dominant lookahead component was found -- see Section 7's confound note for the residual uncertainty a bundled smoothing/seasonal-adjustment comparison can't fully rule out.


## 9. Persist results

In [8]:
results = {
    "national_turning_points": national_turning_point_results,
    "county_turning_points": county_turning_point_results,
    "lag_correlations": lag_correlations,
    "asymmetric_toward_past": asymmetric_toward_past,
    "smoothed_leads_nationally": smoothed_leads_nationally,
    "smoothed_leads_county": smoothed_leads_county,
    "verdict": verdict,
}

os.makedirs("../outputs", exist_ok=True)
with open("../outputs/zhvi_smoothing_check.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved ../outputs/zhvi_smoothing_check.json")


Saved ../outputs/zhvi_smoothing_check.json


## 10. Findings

- **This is the first time this check has been run against this project's own
  downloaded ZHVI file**, rather than relying on the sibling `Housing Price
  Decline` project's result for the same question. The verdict printed in
  Section 8 and saved to `../outputs/zhvi_smoothing_check.json` is this
  project's own evidence.
- **National turning points** (Section 4) and **a data-selected county's
  turning points** (Section 5) are the stronger evidence: a smoothed series
  that ever leads its own raw version at a real turning point would be the
  specific signature of a lookahead component, and neither seasonal
  adjustment nor chance would explain that on its own.
- **The full-history correlation sweep** (Section 6) is the secondary,
  weaker check -- it shows whether the filter leans trailing across the
  whole 2000-2026 history, not that it contains zero forward weight.
- **The bundled raw/NSA vs. smoothed/SA confound** (Section 7) means this
  test can't fully separate the smoothing effect from the seasonal-adjustment
  effect. That's a known limitation of what Zillow makes available, not
  something this notebook controls for.
- **Not done here:** this notebook only checks the smoothing filter's
  direction. It doesn't re-derive or change any feature in
  `src/housing_risk/features.py` -- if Section 8's verdict comes back
  anything other than "trailing, not centered," that's the trigger to go
  back and re-examine the momentum features, not something this notebook
  fixes on its own.
